# 00 - Run All Notebooks

**Notebook version:** v24 -- 2026-08-02

Runs `01` through `06` in order, one command, no manual open-run-repeat needed.
Each notebook still exists as its own independent, reviewable file -- this just
automates running through all of them sequentially.

**Before running:** this executes real model training (RQ1's TabNet CV loop
especially) across all six notebooks back to back -- expect this to take a
while. If you're only iterating on one notebook, open and run that one
directly instead; use this when you want a full, clean, start-to-finish run.

**What this does NOT do:** it does not replace reviewing each notebook's
actual output -- after it finishes, open the notebooks (or the HTML files it
generates) to check the real results, same as you would running them
individually.

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

## Run each notebook in sequence

Each notebook's own trailing HTML-export cell now detects when there's no
live Colab frontend attached (the case during this kind of unattended,
sequential execution) and skips gracefully instead of raising -- so
execution here runs with normal, strict error handling (no
`--allow-errors`). A real failure anywhere in a notebook stops that
notebook's run and is reported as FAILED below, instead of being silently
swallowed -- this was a real bug found in practice: `--allow-errors` used
to mask genuine cell failures (e.g. a SHAP computation error) and report
the notebook as OK anyway, which made the run summary untrustworthy.


In [ ]:
import subprocess
import time

NOTEBOOKS_IN_ORDER = [
    "01_data_screening.ipynb",
    "02_modeling_rq1.ipynb",
    "03_shap_analysis_rq2.ipynb",
    "04_feature_reduction_rq3.ipynb",
    "05_attention_comparison_rq4.ipynb",
    "06_anomaly_safety_net.ipynb",
]

run_log = []

for nb in NOTEBOOKS_IN_ORDER:
    print(f"--- Running {nb} ---")
    start = time.time()

    result = subprocess.run(
        [
            "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
            "--ExecutePreprocessor.timeout=1800",
            nb,
        ],
        capture_output=True, text=True,
    )
    elapsed = time.time() - start

    success = result.returncode == 0
    run_log.append({"notebook": nb, "success": success, "seconds": round(elapsed, 1)})

    if success:
        print(f"{nb} finished in {elapsed:.0f}s\n")
    else:
        print(f"{nb} FAILED after {elapsed:.0f}s -- stderr below:")
        print(result.stderr)
        print(f"Stopping here; {nb} and any notebooks after it were not completed.\n")
        break

print("=== Run summary ===")
for entry in run_log:
    status = "OK" if entry["success"] else "FAILED"
    print(f"  [{status}] {entry['notebook']} ({entry['seconds']}s)")

## Export every successfully-run notebook to HTML

Runs directly against the now-populated `.ipynb` files on disk -- this is a
plain, non-interactive `nbconvert --to html`, not the live-frontend trick
each notebook's own export cell uses, so it works correctly in this batch
context.

In [ ]:
html_files = []

for entry in run_log:
    if not entry["success"]:
        continue
    nb = entry["notebook"]
    html_name = nb.replace(".ipynb", ".html")
    result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", nb, "--output", html_name],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        html_files.append(html_name)
        print(f"Exported {html_name}")
    else:
        print(f"Failed to export {nb} to HTML:")
        print(result.stderr)

print(f"\n{len(html_files)} HTML file(s) generated: {html_files}")

## Download all HTML exports as one zip (optional, Colab only)

In [ ]:
if IN_COLAB and html_files:
    import zipfile

    zip_name = "all_notebook_exports.zip"
    with zipfile.ZipFile(zip_name, "w") as zf:
        for html_file in html_files:
            zf.write(html_file)

    from google.colab import files
    files.download(zip_name)
    print(f"Downloading {zip_name} ({len(html_files)} file(s))")
else:
    print("Not in Colab, or no HTML files were generated -- skipping download.")

## Download real data/artifacts to push back to GitHub

The `data/raw/`, `data/model_ready/`, and `data/artifacts/` folders on
GitHub currently hold only placeholder files -- Colab sessions are
ephemeral, so nothing generated during a run syncs back to the repo
automatically. This cell zips up everything actually generated in this
run and triggers a download; unzip it locally and upload its contents
into the matching `data/` folders on GitHub (same overwrite process used
for code changes) to make the repository's data genuinely reproducible,
not just described as such.

In [ ]:
import os
import zipfile

if IN_COLAB:
    data_root = "../data"
    zip_name = "secom_data_and_artifacts.zip"

    with zipfile.ZipFile(zip_name, "w") as zf:
        file_count = 0
        for subfolder in ["raw", "model_ready", "artifacts"]:
            folder_path = os.path.join(data_root, subfolder)
            if not os.path.isdir(folder_path):
                continue
            for fname in os.listdir(folder_path):
                if fname == "PLACEHOLDER.md":
                    continue  # don't re-upload the placeholder itself
                fpath = os.path.join(folder_path, fname)
                if os.path.isfile(fpath):
                    zf.write(fpath, arcname=os.path.join("data", subfolder, fname))
                    file_count += 1

    if file_count > 0:
        from google.colab import files
        files.download(zip_name)
        print(f"Downloading {zip_name} ({file_count} real data/artifact files, "
              f"placeholders excluded)")
    else:
        print("No real data files found -- did the notebooks actually run "
              "successfully above? Nothing to download.")
else:
    print("Not running in Colab -- skipping download. Data files are already "
          "on local disk under data/raw/, data/model_ready/, data/artifacts/.")
